<a href="https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import urllib.request

# Download dataset from the GitHub repository
url = "https://raw.githubusercontent.com/fatimaali123-ai/flyrank-internship/main/data/raw/content_refresh_anonymized.csv"
local_path = "/content/content_refresh_anonymized.csv"

urllib.request.urlretrieve(url, local_path)

df = pd.read_csv(local_path)

# Target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# Fields excluded from predictive features
EXCLUDED = {
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
}

# Numeric feature vector
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in EXCLUDED]

X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

# Fill missing numeric values
X_filled = X.fillna(X.median(numeric_only=True))

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Numeric features used:", len(feature_cols))
print("Feature vector shape:", X_filled.shape)

print("\nFeatures:")
print(feature_cols)

print("\nTarget distribution:")
print(y.value_counts())


Rows: 30000
Columns: 45
Numeric features used: 29
Feature vector shape: (30000, 29)

Features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


### Feature vector

I build a numeric feature vector from fields that are available before the review decision. Identifier fields and target-derived fields are excluded. Numeric missing values are handled with training-safe median imputation during modeling.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

The feature vector contains numeric search, traffic, engagement, content-age, freshness, CTR, position, and content-size signals.

Missing numeric values are handled with median imputation. In the final modeling workflow, medians are calculated from the training data only.

Pseudonymous identifiers are used only for grouping or auditing and are not predictive features.

The intended prediction moment is the point at which the content item is being prioritized for human review. Fields representing the observed target or its direct outcome are not available to the model.

In [7]:
# Feature availability and missing-value audit

feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [df[c].dtype for c in feature_cols],
    "missing_count": [df[c].isna().sum() for c in feature_cols],
    "missing_pct": [df[c].isna().mean() * 100 for c in feature_cols],
})

print("Feature audit:")
display(feature_notes)

print("\nCategorical columns excluded from numeric feature vector:")
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(categorical_cols)


Feature audit:


,feature,dtype,missing_count,missing_pct
0,search_volume,float64,2468,8.226667
1,competition,float64,2468,8.226667
2,cpc,float64,2468,8.226667
3,word_count,float64,7699,25.663333
4,char_count,float64,7699,25.663333
5,impressions_90d,int64,0,0.000000
6,clicks_90d,int64,0,0.000000
7,pageviews_90d,int64,0,0.000000
8,sessions_90d,int64,0,0.000000
9,users_90d,int64,0,0.000000



Categorical columns excluded from numeric feature vector:
['content_id', 'client_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage hunt

I explicitly test for fields that could reveal the target directly or indirectly.

The main risks are label-derived fields such as `trend_direction` and `trend_pct`, identifiers that could encode client-specific patterns, and any fields representing information that would only be known after the prediction point.

The target is created from `trend_direction`, so `trend_direction` and `trend_pct` are excluded from the feature vector.

In [8]:
# Leakage checks

label_derived = ["trend_direction", "trend_pct"]

print("Target-derived fields present:")
for col in label_derived:
    print(f"{col}: {col in df.columns}")

print("\nTarget-derived fields in feature vector:")
for col in label_derived:
    print(f"{col}: {col in feature_cols}")

print("\nIdentifier fields in feature vector:")
for col in ["content_id", "client_id"]:
    print(f"{col}: {col in feature_cols}")

# Direct check
assert "trend_direction" not in feature_cols
assert "trend_pct" not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols

print("\nLEAKAGE CHECK PASSED")
print("No target-derived fields or pseudonymous identifiers are used as model features.")

Target-derived fields present:
trend_direction: True
trend_pct: True

Target-derived fields in feature vector:
trend_direction: False
trend_pct: False

Identifier fields in feature vector:
content_id: False
client_id: False

LEAKAGE CHECK PASSED
No target-derived fields or pseudonymous identifiers are used as model features.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded fields

| Field | Reason for exclusion |
|---|---|
| `content_id` | Pseudonymous identifier; not a meaningful predictive signal |
| `client_id` | Pseudonymous grouping identifier; used for grouped validation, never as a feature |
| `trend_direction` | Directly defines the target and would leak the label |
| `trend_pct` | Outcome-derived field and therefore excluded to reduce leakage risk |

The exclusions are intentional: identifiers can be used for grouping/auditing, while label-derived fields must not enter the predictive feature vector.

In [9]:
# Final exclusion audit

exclusion_reasons = {
    "content_id": "Pseudonymous identifier; not a predictive feature.",
    "client_id": "Grouping identifier; used for validation only.",
    "trend_direction": "Directly used to construct the target.",
    "trend_pct": "Outcome-derived field; leakage risk."
}

exclusion_audit = pd.DataFrame(
    [
        {"field": field, "reason": reason}
        for field, reason in exclusion_reasons.items()
    ]
)

display(exclusion_audit)

print("\nFinal feature count:", len(feature_cols))
print("Excluded fields:", list(exclusion_reasons.keys()))


,field,reason
0,content_id,Pseudonymous identifier; not a predictive feat...
1,client_id,Grouping identifier; used for validation only.
2,trend_direction,Directly used to construct the target.
3,trend_pct,Outcome-derived field; leakage risk.



Final feature count: 29
Excluded fields: ['content_id', 'client_id', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.